In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DoubleType, IntegerType
from delta.tables import DeltaTable

spark.sql("USE CATALOG ecommerce_catalog")

orders_bronze = spark.read.table("bronze.orders_raw")
orders_bronze.count()

In [0]:
orders_bronze.printSchema()
orders_bronze.describe().display()

In [0]:
orders_bronze.select("order_date").distinct().limit(50).display()

In [0]:
orders_stage = (
    orders_bronze
    .withColumn("unit_price", F.regexp_replace("unit_price", r"[^\d.]", "").cast("decimal(10, 2)"))
    .withColumn("total_amount", F.regexp_replace(F.regexp_replace("total_amount", r"[^\d.]", ""), r"^\.+", "").cast("decimal(10,2)"))
    .withColumn("discount_applied", F.regexp_replace("discount_applied", r"[^\d.]", "").cast("decimal(10,2)"))
    .withColumn("quantity", F.col("quantity").cast("int"))
)

In [0]:
orders_stage.printSchema()

In [0]:
#orders_stage = orders_stage.withColumn("order_date_parsed", F.to_date("order_date"))
#orders_stage.filter("order_date is not null and order_date is null").select("order_date").distinct().display()

In [0]:
date_candidates = [
    F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd HH:mm:ss')"),
    F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd')"),
    F.expr("try_to_timestamp(order_date, 'dd/MM/yyyy')"),
    F.expr("try_to_timestamp(order_date, 'dd-MM-yyyy')"),
    F.expr("try_to_timestamp(order_date, 'MM/dd/yyyy')"),
    F.expr("try_to_timestamp(order_date, 'yyyy/MM/dd HH:mm:ss')"),
    F.expr("try_to_timestamp(order_date, 'dd MM yyyy')"),
    F.expr("try_to_timestamp(order_date, 'dd MMM yyyy')"),
]
order_date_clean = F.coalesce(*date_candidates)

created_at_candidates = [
    F.expr("try_to_timestamp(created_at, 'yyyy-MM-dd HH:mm:ss')"),
    F.expr("try_to_timestamp(created_at, 'yyyy-MM-dd')"),
    F.expr("try_to_timestamp(created_at, 'dd/MM/yyyy')"),
    F.expr("try_to_timestamp(created_at, 'dd-MM-yyyy')"),
    F.expr("try_to_timestamp(created_at, 'MM/dd/yyyy')"),
    F.expr("try_to_timestamp(created_at, 'yyyy/MM/dd HH:mm:ss')"),
    F.expr("try_to_timestamp(created_at, 'dd MM yyyy')"),
    F.expr("try_to_timestamp(created_at, 'dd MMM yyyy')"),
]
created_at_clean = F.coalesce(*created_at_candidates)

In [0]:
orders_stage = orders_stage.withColumn("order_date_clean", F.coalesce(*date_candidates))
orders_stage = orders_stage.withColumn("created_at_clean", F.coalesce(*created_at_candidates))

In [0]:
orders_stage.select("order_date", "order_date_clean").display()

In [0]:
bad_dates = orders_stage.filter("order_date is not null and order_date_clean is null")
bad_dates.write.format("delta").mode("append").saveAsTable("silver.orders_quarantine")

orders_stage = orders_stage.filter(
    "order_date is null or order_date_clean is not null"
)

In [0]:
bad_dates = orders_stage.filter("created_at is not null and created_at_clean is null")
bad_dates.write.format("delta").mode("append").saveAsTable("silver.created_quarantine")

orders_stage = orders_stage.filter(
    "created_at is null or created_at_clean is not null"
)

In [0]:
orders_stage.filter("order_status != 'Returned' and return_reason is not null").count()

In [0]:
orders_stage.filter("order_status = 'Returned' and return_reason is null").count()

In [0]:
orders_stage = (orders_stage
                .withColumn("payment_mode", F.upper(F.trim("payment_mode")))
                .withColumn("order_status", F.initcap(F.trim("order_status")))
                .withColumn("delivery_city", F.initcap(F.trim("delivery_city")))
                .withColumn("customer_name", F.trim("customer_name"))
                .withColumn("customer_email", F.lower(F.trim("customer_email")))
)

In [0]:
orders_stage = orders_stage.withColumn(
    "amount_check",
    F.abs(F.col("total_amount") - ((F.col("unit_price") * F.col("quantity")) - F.col("discount_applied")))
)
orders_stage.filter("amount_check > 1").count()


In [0]:
orders_stage.filter("amount_check < 1").select(
    "order_id", "unit_price", "quantity", "discount_applied", "total_amount", "amount_check", "coupon_code"
).orderBy(F.desc("amount_check")).limit(40).display()

In [0]:
total_rows = orders_stage.count()
failed_rows = orders_stage.filter("amount_check >=1").count()
print(f"Failed: {failed_rows}/{total_rows} = {failed_rows/total_rows: 2%}")

In [0]:
orders_silver = orders_stage.drop("amount_check")

In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy("order_id").orderBy(F.col("created_at_clean").desc())

orders_silver = (orders_silver.withColumn("_rn", F.row_number().over(w))
                 .filter("_rn = 1")
                 .drop("_rn")
    
    )

In [0]:
orders_silver.printSchema()
orders_silver.count()


In [0]:
orders_silver.groupBy("order_id").count().filter("count > 1").count()

In [0]:
spark.table("silver.orders").printSchema()

In [0]:
orders_silver.printSchema()

In [0]:
orders_silver = (
    orders_stage.drop("order_date", "created_at", "amount_check", "file_name", "file_size")
    .withColumnRenamed("order_date_clean", "order_date")
    .withColumnRenamed("created_at_clean", "created_at")
)

In [0]:
orders_silver.printSchema()

In [0]:
orders_silver = orders_silver.drop("order_date_parsed")

In [0]:
spark.sql("DROP TABLE IF EXISTS silver.orders")
orders_silver.write.format("delta").mode("overwrite").saveAsTable("silver.orders")

In [0]:
orders_silver.limit(10).display()